# Reflow, step by step

The same run the `reflow` CLI performs, with each stage left open so the
intermediate models can be inspected.

Cells run in order: later ones use `model`, `pruned`, `reflowed` and `batches`
defined earlier. For a one-liner instead, use `reflow.run_experiment(...)`.

In [ ]:
import copy

from reflow import (
    cache_batches,
    collect_activation_variance,
    evaluate,
    load_dataset,
    load_model,
    prune_model,
    reflow,
    resolve_device,
    sparsity,
    variance_ratios,
)

MODEL = "mobilenet"       # see `reflow --list-models`
SPARSITY = 0.8
CALIBRATION_BATCHES = 50
BATCH_SIZE = 128

device = resolve_device()
device

## Load the model and the data

`spec` carries the model's dataset and the top-1 accuracy its weights are
published with, so the dense measurement below has something to check against.

In [ ]:
model, spec = load_model(MODEL, device=device)
calibration_loader, test_loader = load_dataset(
    spec.dataset, batch_size=BATCH_SIZE, num_workers=8)

print(spec.description, "|", spec.dataset, "|", spec.reference_top1)

## Stage 1 — dense accuracy

In [ ]:
dense_accuracy = evaluate(model, test_loader, device)
print(f"dense: {dense_accuracy:.2f}%")

## Stage 2 — one-shot magnitude pruning

Global ranking: every prunable weight competes against every other one, so the
sparsity budget lands wherever magnitudes are smallest.

In [ ]:
pruned = prune_model(copy.deepcopy(model), SPARSITY)
pruned_accuracy = evaluate(pruned, test_loader, device)
print(f"pruned: {pruned_accuracy:.2f}%  (measured sparsity {sparsity(pruned):.4f})")

## Stage 3 — reflow

The calibration batches are drawn once and cached, so reflow and the variance
measurement below both see identical inputs.

In [ ]:
batches = cache_batches(calibration_loader, CALIBRATION_BATCHES)

reflowed = copy.deepcopy(pruned)
info = reflow(reflowed, batches, device)

reflow_accuracy = evaluate(reflowed, test_loader, device)
print(info)
print(f"pruned + reflow: {reflow_accuracy:.2f}%  "
      f"({reflow_accuracy - pruned_accuracy:+.2f} points)")

## Where the accuracy went: the variance ratio

$\eta_\ell = \mathrm{Var}_{\text{pruned}}(Z_\ell) / \mathrm{Var}_{\text{dense}}(Z_\ell)$
per normalization layer, in forward order. The pruned curve decays with depth;
after reflow it sits back on 1.0.

In [ ]:
measure_on = batches[:16]

dense_stats, order = collect_activation_variance(model, measure_on, device)
pruned_stats, _ = collect_activation_variance(pruned, measure_on, device)
reflow_stats, _ = collect_activation_variance(reflowed, measure_on, device)

ratios = {
    "pruned": variance_ratios(dense_stats, pruned_stats, order),
    "pruned + reflow": variance_ratios(dense_stats, reflow_stats, order),
}

print(f"{len(order)} layers; final-layer eta: "
      f"{ratios['pruned']['output'][-1]:.3f} -> "
      f"{ratios['pruned + reflow']['output'][-1]:.3f}")

In [ ]:
import os

from IPython.display import Image, display

from reflow.plotting import plot_accuracy, plot_variance_ratios

# Written under results/ (gitignored) rather than the repo root, so running the
# notebook does not litter the working tree.
out_dir = "results/notebook"
os.makedirs(out_dir, exist_ok=True)

title = f"{MODEL} - {SPARSITY:.0%} sparsity"
variance_png = plot_variance_ratios(
    ratios, f"{out_dir}/variance_ratio.png", title=title)
accuracy_png = plot_accuracy(
    {"dense": dense_accuracy,
     "pruned": pruned_accuracy,
     "pruned + reflow": reflow_accuracy},
    f"{out_dir}/accuracy.png", title=title)

display(Image(variance_png), Image(accuracy_png))